### Final Project ###


In central forces the main goal as we progressed throughout the term was to use schrodinger's wave equation to model the hydrogen atom and how each orbital would look for a given principle quantum number (n), orbital angular momentum(l), and magnetic quantum number(m). Our main question we proposed was how do these probability distributiion orbitals look when we plot them out for varying n,l or m values. 

### Background ###

Schrodingers wave equation is defined as 
$$
i\hbar \frac{\partial}{\partial t}\psi(\mathbf{r},t)
=
\left(-\frac{\hbar^2}{2m}\nabla^2 + V(\mathbf{r},t)\right)\psi(\mathbf{r},t)
=
E\psi(\mathbf{r},t)
$$


For our purposes we will only consider the time-independent solution so we can simplify our equation to become 

$$
\left(-\frac{\hbar^2}{2m}\nabla^2 + V(\mathbf{r})\right)\psi(\mathbf{r})
=
E\psi(\mathbf{r})
$$


In order to solve this PDE we can seperate each variable such that $$ \psi(\mathbf{r}) =  \psi(r,\theta,\phi) = R(r) Y(\theta , \phi) $$

For a hydrogen atom we will considered the electrostatic potential between proton in the nuecleus and the electron orbiting the nuecleus. If we were to fix our reference frame at the proton then we can write our schrodinger equation in terms of our seperated variables as 


$$\frac{-\hbar^2}{2\mu}\left({\frac{Y(\theta , \phi)}{r^2}\frac{d}{dr}r^2 \frac{dR(r)}{dr} + R(r)\frac{\hat{L^2}}{\hbar^2}Y(\theta , \phi)}\right)-\frac{Ke^2}{r}R(r)Y=ER(r)Y(\theta , \phi)$$

where $\mu $ is the reduced mass, $K$ is the Coulombs constant, and $\hat{L^2}$ is the total angular momentum operator. Solving this by splitting this into two ODEs we get a solution in terms on n, l,and m. So then our wave function looks like $$ \psi_{n,l,m}(\mathbf{r}) =  \psi_{n,l,m}(r,\theta,\phi) = R(r)_n Y_{l}^{m}(\theta , \phi) $$


So our wave function has unique solutions for given n,l, and m. Thus our probability distribution of our orbital will vary for different n, l, and m values. Analytically solving for our wave function we find that we have conditions that state $n \geq l$ and $l \geq m$. 

Some major assumptions we are making for our model is first that we are looking at a system where our reference frame is centered at the proton. We also assume that our system is conserved so no energy is being lost. For our wave function we are considered only specific n, l, and m's for our wave function and not a superposition of these. This will also take care of the time dependence of our wave function allows us to see constant probability distributions.  

Solving for the radial solution factor $R(r)$ yields

$$R_{n,\ell} (r) = -\sqrt{\left(\frac{2Z}{na_0}\right)^3 \frac{(n - \ell - 1)!}{2n[(n + \ell)!]^3}} e^{-Zr/na_0} \left(\frac{2Zr}{na_0}\right)^\ell L_{n+\ell}^{2\ell + 1} \big(2Zr/na_0\big)$$



In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.special as sp
import math
import ipywidgets as widgets
from IPython.display import display

#Bohr radius
a0_num = 1.0

For simplicity, the Bohr radius is set to be 1.

In [2]:
def radial_hydrogen(r, n, l):
    rho = 2 * r / (n * a0_num)

    #Calculates normalization factor
    norm_factor = np.sqrt((2/(n*a0_num))**3 *math.factorial(n-l-1) /(2*n*math.factorial(n+l)))

    #Calculates Laguerre polynomial
    laguerre = sp.genlaguerre(n-l-1, 2*l+1)
    
    #Returns radial solution for spherical harmonic
    return norm_factor * np.exp(-rho/2) * rho**l * laguerre(rho)

The "radial_hydrogen" function calculates the radial solution factor of the electron orbital solutions for the hydrogen atom, having parameters of radial distance "r", principle quantum number "n", and azimuthal quantum number "l". It calculates the normalization constant "norm_factor" using the input quantum numbers. The associated Laguerre polynomial is then calculated using these quantum numbers, and the normalization factor is then multiplied to the exponential and the Laguerre polynomial and returned.

In [3]:
def plot_hydrogen(n, l, m):
    
    #Checking the dependencies of the quantum numbers
    if l >= n or abs(m) > l:
        print("Invalid quantum numbers")
        return
    
    #Creating our X and Z axis and turning them into 2D arrays of their values
    x = np.linspace(-20, 20, 400)
    z = np.linspace(-20, 20, 400)
    X, Z = np.meshgrid(x, z)

    #Calculating the radius and angle from the origin at each point in the grid
    R = np.sqrt(X**2 + Z**2)
    TH = np.arccos(Z / (R + 1e-12))

    #Solving for the radial component and the spherical harmonic for the given quantum numbers
    Rad = radial_hydrogen(R, n, l)
    Y = sp.sph_harm_y(l, m, TH, 0)

    #Pluggin in our components to find our wave function and from there calculating the probability density by taking the norm squared
    psi = Rad * Y
    prob_density = np.abs(psi)**2

    #Plotting the probability density on a contour plot
    plt.figure(figsize=(7,6))
    plt.contourf(X, Z, prob_density, levels=80, cmap="viridis")
    
    #Setting colorbar and plot components
    cbar = plt.colorbar()
    cbar.set_label(r"$|\psi|^2$", rotation=0, labelpad=20)
    plt.title(f"Hydrogen orbital (n={n}, l={l}, m={m})")
    plt.xlabel("x")
    plt.ylabel("z", rotation = 'horizontal')
    plt.axis("equal")
    plt.show()

The "plot_hydrogen" function visualizes the solutions to the hydrogen atom orbitals. It first checks if the conditions that the quantum numbers must satisfy ($n \ge \ell$ and $\ell \ge m$). It then creates arrays "x" and "z" to 

In [ ]:
#Interactive quantum number sliders, updates the plot whenever a value is changed

n_slider = widgets.IntSlider(value=2, min=1, max=6, description='n')
l_slider = widgets.IntSlider(value=1, min=0, max=5, description='l')
m_slider = widgets.IntSlider(value=0, min=-5, max=5, description='m')

def update(n, l, m):
    plot_hydrogen(n, l, m)

ui = widgets.VBox([n_slider, l_slider, m_slider])

out = widgets.interactive_output(update,{'n': n_slider, 'l': l_slider, 'm': m_slider})

display(ui, out)

Output()

We also solved the differential equation describing the Hydrogen atom numerically. To do this, we used separation of variables to turn the partial differential equation $\hat{H}\psi=E\psi$ into three ordinary differential equations. The equations that result from this process are
$$ -\frac{\partial}{\partial r} (r^2 \frac{\partial R}{\partial r}) - \frac{2\mu K e^2 r}{\hbar^2} R+A R=E\frac{2\mu r^2}{\hbar^2}R $$
$$ -\frac{\partial}{\partial \theta} (\sin \theta\frac{\partial \Theta}{\partial \theta}) + \frac{B}{\sin\theta}\Theta = A\sin\theta\Theta $$
$$ -\frac{\partial^2 \Phi}{\partial \phi^2} = B\Phi $$
Here we have put the differential equations in Sturm-Liouville form, so that the differential operators are Hermitian. This will help guarantee that we get real eigenvalues and orthogonal eigenstates.

We want to solve these numerically through finite difference methods by approximating the differential operators as matrices. Starting with the $\phi$ equations, this is a simply a second derivative, and so can we can approximate $\frac{\partial^2\Phi}{\partial\phi^2}$ by a matrix of the form
$$ \begin{pmatrix}
-2 & 1 & 0 & \cdots & 1 \\
1 & -2 & 1 & \cdots & 0 \\
0 & 1 & -2 & \cdots & 0 \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & 0 & 0 & \cdots & -2
\end{pmatrix} . $$
Note that the 1's in the corners enforce the periodic boundary condition of $\phi$, as it treats the first and final positions as adjacent for the purpose of the derivative.

The differential operator for the $\theta$ equation is slightly more complex to approximate. A symmetric single derivative with finite difference methods can be represented as
$$ \frac{\partial}{\partial\theta} \Theta(\theta_i) \approx \frac{\Theta(\theta_{i+1/2}) - \Theta(\theta_{i-1/2})}{\Delta\theta} $$
Note that $\Theta$ is not defined at fractional indicies of $\theta$, so this equation is not possible to implement, but it serves as a useful mathematical intermediate. To construct the term $\sin\theta\frac{\partial\Theta}{\partial\theta}$, we can simply multiply this formula by $\sin\theta$.
$$ \sin\theta\frac{\partial\Theta}{\partial\theta} \approx \sin\theta_{i} \frac{\Theta(\theta_{i+1/2}) - \Theta(\theta_{i-1/2})}{\Delta\theta} $$
Taking the derivative of this in a similar manner as before, we find that
$$ \frac{\partial}{\partial\theta} (\sin\theta\frac{\partial\Theta}{\partial\theta}) \approx \frac{\sin\theta_{i+1/2} \frac{\Theta(\theta_{i+1}) - \Theta(\theta_{i})}{\Delta\theta} - \sin\theta_{i-1/2} \frac{\Theta(\theta_{i}) - \Theta(\theta_{i-1})}{\Delta\theta}}{\Delta\theta} $$
$$ \frac{\partial}{\partial\theta} (\sin\theta\frac{\partial\Theta}{\partial\theta}) \approx \frac{1}{(\Delta\theta)^2} (\sin\theta_{i+1/2} \Theta(\theta_{i+1}) - (\sin\theta_{i+1/2} + \sin\theta_{i-1/2})\Theta(\theta_{i}) + \sin\theta_{i-1/2} \Theta(\theta_{i-1})) $$
The discretized function $\Theta(\theta_i)$ is no longer being evaluated at fractional indicies of $\theta$, so this equation is valid to implement. As a linear equation, we can put it in matrix form as
$$ \begin{pmatrix}
\ddots & \vdots & \vdots & \vdots & \\
\cdots & -\sin\theta_{i-3/2}-\sin\theta_{i-1/2} & \sin\theta_{i-1/2} & 0 & \cdots \\
\cdots & \sin\theta_{i-1/2} & -\sin\theta_{i-1/2}-\sin\theta_{i+1/2} & \sin\theta_{i+1/2} & \cdots \\
\cdots & 0 & \sin\theta_{i+1/2} & -\sin\theta_{i+1/2}-\sin\theta_{i+3/2} & \cdots \\
& \vdots & \vdots & \vdots & \ddots
\end{pmatrix} $$
From this, we can see that the matrix is Hermitian, which is a good indication that we have represented the operator in an accurate way. Finally, the eigenvalue for the $\theta$ equation is $A$, which is multiplied by $\sin\theta$ in addition to the eigenstate. This means that we need to solve the generalized eigenvalue equation $D v = \lambda W v$, where $D$ and $W$ are both matrices. Python's `scipy` package has a solver for this.

We can now move our attention to the $r$ equation. The boundary conditions for $r$ are slightly different from the previous equations since we want $R(r)$ to go to 0 as $r\rightarrow\infty$. Therefore we want to sample $R(r)$ at both small and large values of $r$. To do this, we can make the substitution $r=r_0\tan(\rho)$, and sample $\rho$ on the interval $[0,\frac{\pi}{2}]$. After making this substitution, the differential eqaution for $r$ becomes
$$ -\frac{\partial}{\partial\rho} (r_0 \sin^2(\rho) \frac{\partial R}{\partial\rho}) - \frac{2\mu K e^2 r_0^2}{\hbar^2} \frac{\sin(\rho)}{\cos^3(\rho)} R + \frac{A r_0}{\cos^2(\rho)} R = E \frac{2\mu r_0^3}{\hbar^2} \frac{\sin^2(\rho)}{\cos^4(\rho)} R $$
At this point, we can approximate the differential equation as an eigenvalue equation in the same way as we did for $\theta$.

Once we have these matricies set up, we can find the eigenvalues and eigenstates by simply calling `scipy.linalg.eigh`, which solves the generalized eigenvalue equation for Hermitian matrices.

In [5]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg

N = 500
RADIUS_FOCUS = 5.292e-11

PHI_CUTTOFF = 25
SPHERICAL_HARMONIC_CUTOFF = 64
HYDROGEN_ORBITAL_CUTOFF = 100

HBAR = 1.0546e-34
K = 1/(4*np.pi * 8.8542e-12)
MU = 9.1094e-31
ECHARGE = 1.6022e-19

def PhiOperator():
  dphi = 2*np.pi / (N)
  dif = np.diag(np.ones(N)) - np.diag(np.ones(N-1), k=-1)
  corner_one = np.zeros((N,N)); corner_one[0][N-1] = 1
  periodic_dif = dif - corner_one
  phi_operator = (periodic_dif.T + periodic_dif) * (1/dphi**2)
  return phi_operator

def ThetaOperator(B):
  dtheta = np.pi / (N+1)
  main_diagonal = np.diag([np.sin((3/2) * dtheta), *[
      (np.sin((n+1/2) * dtheta) + np.sin((n-1/2) * dtheta))
    for n in range(2,N)], np.sin(((N) - 1/2) * dtheta)])
  top_off_diagonal = np.diag([
      np.sin((n+1/2) * dtheta)
    for n in range(1,N)], k=1)
  bottom_off_diagonal = np.diag([
      np.sin((n-1/2) * dtheta)
    for n in range(2,N+1)], k=-1)
  differential_operator = (main_diagonal - bottom_off_diagonal - top_off_diagonal) * (1/dtheta**2)

  B_term = np.diag([
      B / np.sin((n) * dtheta)
    for n in range(1,N+1)])
  theta_operator = differential_operator + B_term

  W = np.diag([
      np.sin((n) * dtheta)
    for n in range(1,N+1)])
  return (theta_operator, W)

def ROperator(A, Z=1):
  r0 = RADIUS_FOCUS
  drho = (np.pi/2) / (N+1)
  main_diagonal = np.diag([r0 * np.sin((3/2) * drho)**2, *[
      (r0 * np.sin((n+1/2) * drho)**2 + r0 * np.sin((n-1/2) * drho)**2)
    for n in range(2,N)], r0 * np.sin(((N) - 1/2) * drho)**2])
  top_off_diagonal = np.diag([
      r0 * np.sin((n+1/2) * drho)**2
    for n in range(1,N)], k=1)
  bottom_off_diagonal = np.diag([
      r0 * np.sin((n-1/2) * drho)**2
    for n in range(2,N+1)], k=-1)
  differential_operator = (main_diagonal - bottom_off_diagonal - top_off_diagonal) * (1/drho**2)

  potential_term = np.diag([
      -(2*MU*K*Z*ECHARGE**2/HBAR**2) * r0**2 * np.sin((n) * drho) / np.cos((n) * drho)**3
    for n in range(1,N+1)])
  A_term = np.diag([
      A * r0 / np.cos((n) * drho)**2
    for n in range(1,N+1)])
  r_operator = differential_operator + potential_term + A_term

  W = np.diag([
      (2 * MU / HBAR**2) * r0**3 * np.sin((n) * drho)**2 / np.cos((n) * drho)**4
    for n in range(1,N+1)])
  return (r_operator, W)


# Solve the phi equation
phi_operator = PhiOperator()
phi_eigenvalues, phi_eigenstates = scipy.linalg.eigh(phi_operator)
phi_eigenstates = phi_eigenstates.transpose()
phi_indicies = np.argsort(phi_eigenvalues)
B = phi_eigenvalues[phi_indicies][:PHI_CUTTOFF]

# Solve the theta equation
theta_operators = [ThetaOperator(b) for b in B]
theta_eig = [scipy.linalg.eigh(op[0], b=op[1]) for op in theta_operators]
theta_eigenstates = np.array([eig[1].transpose() for eig in theta_eig]).reshape((-1,N))
unsorted_AB = np.array([[[b,a] for a in theta_eig[i][0]] for i,b in enumerate(B)]).reshape((-1,2)).transpose()
theta_indicies = np.lexsort(np.round(unsorted_AB, 2))
A = unsorted_AB[1][theta_indicies][:SPHERICAL_HARMONIC_CUTOFF]
phi_indicies_for_A = phi_indicies[theta_indicies // N]
B_sorted_for_A = phi_eigenvalues[phi_indicies_for_A][:SPHERICAL_HARMONIC_CUTOFF]

# Solve the r equation
r_operators = [ROperator(a) for a in A]
r_eig = [scipy.linalg.eigh(op[0], b=op[1]) for op in r_operators]
r_eigenstates = np.array([eig[1].transpose() for eig in r_eig]).reshape((-1,N))
unsorted_EA = np.array([[[B_sorted_for_A[i], a, 1/ECHARGE*e] for e in r_eig[i][0]] for i,a in enumerate(A)]).reshape((-1,3)).transpose()
r_indicies = np.lexsort(np.round(unsorted_EA, 2))

# Gather the eigenvalues
E = unsorted_EA[2][r_indicies][:HYDROGEN_ORBITAL_CUTOFF]
theta_indicies_for_E = theta_indicies[r_indicies // N]
A_sorted_for_E = unsorted_AB[1][theta_indicies_for_E][:HYDROGEN_ORBITAL_CUTOFF]
phi_indicies_for_E = phi_indicies_for_A[r_indicies // N]
B_sorted_for_E = phi_eigenvalues[phi_indicies_for_E][:HYDROGEN_ORBITAL_CUTOFF]


In [6]:
def get_eigenstate_index(n, l, m):
  quantum_numbers = []
  for n_i in range(1,7):
    for l_i in range(0,n_i):
      for m_i in range(0,l_i+1):
        quantum_numbers.append((n_i,l_i,m_i))
        if m_i != 0:
          quantum_numbers.append((n_i,l_i,-m_i))
  idx = quantum_numbers.index((n,l,m))
  return idx

def plot_wavefunction(n, l, m):
  if l >= 7 or l >= n or abs(m) > l:
    print("Invalid quantum numbers")
    return (0,0,0)
  idx = get_eigenstate_index(n, l, m)
  r_values = [0, *(RADIUS_FOCUS * np.tan(np.pi/2 / (N+1) * np.arange(1,N+1))), np.inf]
  r_wavefunction = [0, *(r_eigenstates[r_indicies][idx]), 0]
  theta_values = [0, *(np.pi / (N+1) * np.arange(1,N+1)), np.pi]
  theta_wavefunction = [0, *(theta_eigenstates[theta_indicies_for_E][idx]), 0]
  # phi_values = 2*np.pi / (N) * np.arange(N)
  # phi_wavefunction = phi_eigenstates[phi_indicies_for_E[idx]]

  r0 = 5.292e-11
  x_plotvals = r0 * np.linspace(-20, 20, 100)
  z_plotvals = r0 * np.linspace(-20, 20, 100)
  wavefunction = np.zeros((len(z_plotvals), len(x_plotvals)))
  for i,x in enumerate(x_plotvals):
    for j,z in enumerate(z_plotvals):
      r = np.sqrt(x**2 + z**2)
      theta = np.arccos(z / r)
      closest_r = np.searchsorted(r_values, r)
      closest_theta = np.searchsorted(theta_values, theta)
      wavefunction_at_r = 1/(r_values[closest_r] - r_values[closest_r-1]) * ((r_values[closest_r] - r) * r_wavefunction[closest_r] + (r - r_values[closest_r-1]) * r_wavefunction[closest_r-1])
      wavefunction_at_theta = 1/(theta_values[closest_theta] - theta_values[closest_theta-1]) * ((theta_values[closest_theta] - theta) * theta_wavefunction[closest_theta] + (theta - theta_values[closest_theta-1]) * theta_wavefunction[closest_theta-1])
      wavefunction[j,i] = wavefunction_at_r * wavefunction_at_theta
  
  prob_density = np.abs(wavefunction)**2

  plt.figure(figsize=(7,6))
  plt.contourf(x_plotvals, z_plotvals, prob_density, levels=80, cmap="viridis")

  cbar = plt.colorbar()
  cbar.set_label(r"$|\psi|^2$", rotation=0, labelpad=20)
  plt.title(f"Hydrogen orbital (n={n}, l={l}, m={m})")
  plt.xlabel("x")
  plt.ylabel("z", rotation = 'horizontal')
  plt.axis("equal")
  plt.show()
  
  return (E[idx], A_sorted_for_E[idx], B_sorted_for_E[idx])


In [ ]:
import ipywidgets as widgets
from IPython.display import display

n_slider = widgets.IntSlider(value=2, min=1, max=6, description='n')
l_slider = widgets.IntSlider(value=1, min=0, max=5, description='l')
m_slider = widgets.IntSlider(value=0, min=-5, max=5, description='m')

def update(n, l, m):
    E, A, B = plot_wavefunction(n, l, m)
    print(f"Energy: {round(E,3)} eV")
    print(f"L^2:    {round(A,3)} hbar^2")
    print(f"L_z^2:  {round(B,3)} hbar^2")

ui = widgets.VBox([n_slider, l_slider, m_slider])

out = widgets.interactive_output(update,{'n': n_slider, 'l': l_slider, 'm': m_slider})

display(ui, out)

Output()

### Conclusion ###

The resulting plot of the electron density gives us a good representation of how we expect an electron to behave while bounded to a hydrogen atom. As we change n, l, and m, we can see how the size, shape, and orientation of the oribtals respectively change. As n increases, the energy level of the atom increases which means the electron has more space where it could travel. As l increases, the change in angular momentum creates more possible angular nodes where the electron could travel. And as m increases, the only thing that changes is the orientation of the angular nodes around the given axis which in this case is the proton.

### Reflection ###

We are not considering superpositions of these solution eigenstates, meaning that we are looking at the probability distributions for each eigenstate individually, which are time-independent. Due to cross terms when finding the probability density for a superposition state, the probability density would 